# 01 - Data Loading and Evaluation Metrics
The following cells define the core utility functions for loading all necessary data.
* load_EOD_data: This function is responsible for loading the time-series features for each stock. It's important to note a critical modification made here: while the input features (eod_data) are Z-score normalized, the ground truth returns (ground_truth) are calculated using the raw, un-normalized prices. This ensures that the model's performance is evaluated against the actual market returns, which is crucial for realistic backtesting. It also handles masking for missing data points.

* load_relation_data: This function loads the pre-computed relational matrix between stocks (e.g., based on industry or WikiData links). It also creates a corresponding mask that is used in the attention mechanism to prevent the model from considering connections between unrelated stocks.

* evaluate: This function calculates the performance of a model's predictions. It computes the standard Mean Squared Error (MSE) for regression accuracy. More importantly, it calculates ranking-based metrics like Mean Reciprocal Rank of the Top-ranked stock (mrrt) and a simplified backtesting long-only strategy return (btl), which are better indicators of a model's practical utility in a trading context.

In [1]:
import numpy as np
import os

## Data loading

In [2]:
def load_EOD_data(data_path, market_name, tickers, raw_price_matrix, steps=1):
    eod_data = []
    masks = []
    ground_truth = []
    base_price = [] 
    
    for index, ticker in enumerate(tickers):
        # Z-SCORE NORMALIZED features
        single_EOD = np.genfromtxt(
            os.path.join(data_path, market_name + '_' + ticker + '_1.csv'),
            dtype=np.float32, delimiter=',', skip_header=False
        )
        if market_name == 'NASDAQ':
            single_EOD = single_EOD[:-1, :]

        if index == 0:
            print('single EOD data shape:', single_EOD.shape)
            eod_data = np.zeros([len(tickers), single_EOD.shape[0], single_EOD.shape[1] - 1], dtype=np.float32)
            masks = np.ones([len(tickers), single_EOD.shape[0]], dtype=np.float32)
            ground_truth = np.zeros([len(tickers), single_EOD.shape[0]], dtype=np.float32)
            base_price = np.zeros([len(tickers), single_EOD.shape[0]], dtype=np.float32)

        # Use the raw price matrix to calculate ground truth
        ticker_raw_prices = raw_price_matrix[index]

        for row in range(single_EOD.shape[0]):
            if abs(single_EOD[row][-1] + 1234) < 1e-8:
                masks[index][row] = 0.0
            
            elif row >= steps:
                # Get raw prices for today (row) and yesterday (row - steps)
                price_today = ticker_raw_prices[row]
                price_yesterday = ticker_raw_prices[row - steps]

                # Check for valid prices before calculating return
                if price_yesterday > 1e-8 and price_today > -1000 and price_yesterday > -1000:
                    ground_truth[index][row] = (price_today - price_yesterday) / price_yesterday
                else:
                    # If prices are invalid (0 or placeholder), mask this day out
                    ground_truth[index][row] = 0.0
                    masks[index][row] = 0.0

            # Fill placeholders in feature data
            for col in range(single_EOD.shape[1]):
                if abs(single_EOD[row][col] + 1234) < 1e-8:
                    single_EOD[row][col] = 1.1

        eod_data[index, :, :] = single_EOD[:, 1:]
        base_price[index, :] = single_EOD[:, -1]

    return eod_data, masks, ground_truth, base_price

In [3]:
data_path = '../../data/2013-01-01'
market_name = 'NASDAQ'
tickers_fname = f"{market_name}_tickers_qualify_dr-0.98_min-5_smooth.csv"
steps = 1

In [4]:
tickers = np.genfromtxt(os.path.join(data_path, '..', tickers_fname),
                             dtype=str, delimiter='\\t', skip_header=False)
print('#tickers selected:', len(tickers))

# Load the RAW prices for ground truth calculation in load_EOD_data
raw_price_path = os.path.join(data_path, f'{market_name}_raw_prices.npy')
raw_price_data = np.load(raw_price_path)
if market_name == 'NASDAQ':
    raw_price_data = raw_price_data[:, :-1]
print('Raw prices shape:', raw_price_data.shape)

#tickers selected: 1026
Raw prices shape: (1026, 1245)


In [5]:
eod_data, mask_data, gt_data, price_data = \
            load_EOD_data(data_path, market_name, tickers, raw_price_data, steps)

single EOD data shape: (1245, 6)


In [6]:
# Returns relation encoding and mask
def load_relation_data(relation_file):
    relation_encoding = np.load(relation_file)
    print('relation encoding shape:', relation_encoding.shape) # [Nodes, Nodes, Relation types]
    rel_shape = [relation_encoding.shape[0], relation_encoding.shape[1]] # [Nodes, Nodes]
    # True = no relation, False = at least one relation
    mask_flags = np.equal(np.zeros(rel_shape, dtype=int), np.sum(relation_encoding, axis=2))
    # Where true -> -1e9 (-inf)
    mask = np.where(mask_flags, np.ones(rel_shape) * -1e9, np.zeros(rel_shape))
    return relation_encoding, mask


In [7]:
relation_name='wikidata'
# relation_name='sector_industry'

rname_tail = {'sector_industry': '_industry_relation.npy',
              'wikidata': '_wiki_relation.npy'}

rel_encoding, rel_mask = load_relation_data(
    os.path.join(data_path, '..', 'relation', 'relation', relation_name,
                 market_name + rname_tail[relation_name])
)
print('relation encoding shape:', rel_encoding.shape)
print('relation mask shape:', rel_mask.shape)

relation encoding shape: (1026, 1026, 43)
relation encoding shape: (1026, 1026, 43)
relation mask shape: (1026, 1026)


## Evaluation function

In [8]:
def evaluate(prediction, ground_truth, mask, report=False):
    assert ground_truth.shape == prediction.shape, 'shape mis-match'
    performance = {}
    performance['mse'] = np.linalg.norm((prediction - ground_truth) * mask)**2 / np.sum(mask)

    mrr_top = 0.0
    all_miss_days_top = 0
    # FIX 1: Initialize to 0.0, not 1.0
    bt_long = 1.0
    bt_long5 = 1.0
    bt_long10 = 1.0

    for i in range(prediction.shape[1] - 1):
        
        # --- Use NEXT DAY's ground truth and mask ---
        day_gt = ground_truth[:, i + 1]
        day_mask = mask[:, i + 1]
        
        # --- Use CURRENT DAY's prediction ---
        day_pred = prediction[:, i]
        
        # Find valid stocks for the NEXT day
        valid_indices = np.where(day_mask > 0.5)[0]
        if len(valid_indices) == 0:
            all_miss_days_top += 1
            continue

        # Get ranks for both prediction and ground truth on the valid stocks
        rank_gt_valid = np.argsort(day_gt[valid_indices])[::-1]
        rank_pre_valid = np.argsort(day_pred[valid_indices])[::-1]

        # Get the top-ranked stock according to prediction
        pre_top1_idx = valid_indices[rank_pre_valid[:1]]
        pre_top5_idx = valid_indices[rank_pre_valid[:5]]
        pre_top10_idx = valid_indices[rank_pre_valid[:10]]
        
        # --- MRR Calculation ---
        gt_top1_ticker = valid_indices[rank_gt_valid[0]]
        rank_list = valid_indices[rank_pre_valid]
        position_list = np.where(rank_list == gt_top1_ticker)[0]
        if len(position_list) > 0:
            mrr_top += 1.0 / (position_list[0] + 1)
        else:
            all_miss_days_top +=1


        # Calculate returns using the NEXT DAY's ground truth
        bt_long += np.mean(day_gt[pre_top1_idx])
        bt_long5 += np.mean(day_gt[pre_top5_idx])
        bt_long10 += np.mean(day_gt[pre_top10_idx])

    effective_days = (prediction.shape[1] -1) - all_miss_days_top
    performance['mrrt'] = mrr_top / effective_days if effective_days > 0 else 0
    
    # Pure arithmetic sum (IRR)
    performance['btl'] = bt_long
    performance['btl5'] = bt_long5
    performance['btl10'] = bt_long10
    
    return performance